# Práctica 1. Detector de infieles (parodia) con árboles de decisión

El conjunto de datos es completamente sintético. La variable `infiel` fue inventada para representar estereotipos humorísticos, **no** describe comportamientos reales ni permite juzgar a las personas. La actividad también sirve para ilustrar cómo un algoritmo aprende los sesgos presentes en sus datos.

## Paso 1. Preparación del entorno

En primer lugar, se importan las bibliotecas necesarias:

- **pandas:** organiza y permite consultar los datos.
- **scikit-learn:** divide los datos, entrena el clasificador y calcula las métricas.
- **matplotlib:** representa gráficamente el árbol y la matriz de confusión.

Si alguna biblioteca no está instalada, se puede ejecutar en una celda aparte: `%pip install pandas scikit-learn matplotlib`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

print("Bibliotecas cargadas correctamente")

## Paso 2. Carga del archivo CSV

Se coloca el archivo `dataset_detector_infieles_friki.csv` en la misma carpeta que este notebook. Luego se lee con `pd.read_csv()` y se muestran las primeras cinco filas.

In [ ]:
df = pd.read_csv("dataset_detector_infieles.csv")
print("Filas y columnas:", df.shape)
display(df.head())

## Paso 3. Exploración inicial

Se revisan los nombres de las columnas, la existencia de valores ausentes y la cantidad de ejemplos de cada clase.

En esta práctica, `0` significa **fiel** y `1` significa **infiel**, exclusivamente dentro de la historia

In [ ]:
print("Columnas disponibles:")
print(df.columns.tolist())

print("\nValores faltantes por columna:")
print(df.isna().sum())

print("\nDistribución de clases:")
print(df["infiel"].value_counts().sort_index())

## Paso 4. Selección de las variables de entrada y salida

Para simplificar el primer modelo, se seleccionan **11 variables numéricas** del archivo. Las variables con valores `0` y `1` representan respuestas de tipo no/sí.

- **X:** contiene las características que recibe el algoritmo.
- **y:** contiene la respuesta que el algoritmo debe aprender a predecir.

Se excluyen `id` porque solo identifica registros y `signo_zodiacal` porque es una variable de texto que requeriría una transformación adicional. Las demás columnas no seleccionadas se reservan para futuras prácticas.

In [ ]:
variables = [
    "edad",
    "numero_relaciones_anteriores",
    "horas_sin_responder",
    "oculta_celular",
    "tiene_excusas",
    "le_gusta_regueton",
    "le_gustan_videojuegos",
    "horas_videojuegos_semana",
    "ve_anime",
    "programa_fines_de_semana",
    "salidas_fiesta_mensual"
]

X = df[variables]
y = df["infiel"]

print("Datos de entrada:", X.shape)
print("Etiquetas:", y.shape)
display(X.head())

## Paso 5. División en entrenamiento y prueba

Se reserva el **80 %** de las filas para que el árbol aprenda y el **20 %** restante para comprobar cómo funciona con datos que no utilizó durante el entrenamiento.

La opción `random_state=42` permite repetir la misma división. El parámetro `stratify=y` procura mantener proporciones similares de las clases en los dos conjuntos.

In [ ]:
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Registros para entrenamiento:", len(X_entrenamiento))
print("Registros para prueba:", len(X_prueba))

## Paso 6. Entrenamiento del árbol de decisión

Se construye un `DecisionTreeClassifier`. Un árbol aprende reglas como: «si la cantidad de salidas es superior a cierto valor, tomar una rama; de lo contrario, tomar otra».

- `criterion="gini"` selecciona el índice Gini para buscar divisiones.
- `max_depth=3` limita la profundidad para que el árbol sea fácil de interpretar.
- `min_samples_leaf=5` evita crear hojas con muy pocos registros.

El método `.fit()` realiza el entrenamiento.

In [ ]:
modelo = DecisionTreeClassifier(
    criterion="gini",
    max_depth=3,
    min_samples_leaf=5,
    random_state=42
)

modelo.fit(X_entrenamiento, y_entrenamiento)
print("Árbol de decisión entrenado correctamente")

## Paso 7. Visualización de las reglas del árbol

Se dibuja el árbol para analizar cómo decide. Cada rectángulo representa un nodo.

- La **primera condición**, situada arriba, corresponde a la raíz.
- La rama izquierda representa habitualmente que la condición se cumple (`True`).
- La rama derecha representa que no se cumple (`False`).
- En cada nodo se muestra su `gini`, el número de `samples`, la distribución `value` y la `class` mayoritaria.

Los valores visibles representan únicamente las reglas aprendidas con este dataset ficticio.

In [ ]:
plt.figure(figsize=(19, 9))
plot_tree(
    modelo,
    feature_names=variables,
    class_names=["Fiel (0)", "Infiel (1)"],
    filled=True,
    rounded=True,
    fontsize=9
)
plt.title("Árbol de decisión: detector ficticio")
plt.tight_layout()
plt.show()

## Paso 8. Predicciones con los datos de prueba

Una vez entrenado, el modelo recibe las filas reservadas en el paso 5 y produce una clase para cada una mediante `.predict()`.

Se comparan las predicciones con las etiquetas originales para observar los aciertos y errores.

In [ ]:
predicciones = modelo.predict(X_prueba)

comparacion = pd.DataFrame({
    "real": y_prueba.to_numpy(),
    "prediccion": predicciones
})

display(comparacion.head(10))

## Paso 9. Evaluación del modelo

Se calculan tres resultados:

1. **Exactitud (*accuracy*):** proporción total de registros clasificados correctamente.
2. **Matriz de confusión:** permite identificar aciertos y errores para cada clase.
3. **Precisión, recall y F1-score:** describen el comportamiento por clase, algo que la exactitud por sí sola no muestra.

Como el conjunto de prueba es pequeño, los valores pueden variar si se modifican los datos, el árbol o la división.

In [ ]:
exactitud = accuracy_score(y_prueba, predicciones)
print(f"Exactitud (accuracy): {exactitud:.2%}")
print("\nReporte de clasificación:")
print(classification_report(y_prueba, predicciones, target_names=["Fiel", "Infiel"], zero_division=0))

matriz = confusion_matrix(y_prueba, predicciones, labels=[0, 1])
visualizacion = ConfusionMatrixDisplay(
    confusion_matrix=matriz,
    display_labels=["Fiel (0)", "Infiel (1)"]
)
visualizacion.plot(cmap="Blues", values_format="d")
plt.title("Matriz de confusión: filas reales, columnas predichas")
plt.show()

### ¿Cómo se interpretan las métricas?

Se interpreta la **exactitud** como el porcentaje de registros de prueba que el modelo clasificó correctamente. Por ejemplo, una exactitud de `0.75` significa **75 % de aciertos en ese conjunto**, no una certeza del 75 % para cada predicción.

En la **matriz de confusión**, las filas corresponden a valores reales y las columnas a predicciones:

| | Predice fiel (0) | Predice infiel (1) |
|---|---:|---:|
| Real fiel (0) | Verdaderos negativos | Falsos positivos |
| Real infiel (1) | Falsos negativos | Verdaderos positivos |

Para la clase `infiel (1)`, la **precisión** responde: «de los casos predichos como infiel, ¿qué proporción tenía etiqueta 1?». El **recall** responde: «de los casos etiquetados como infiel, ¿qué proporción detectó el modelo?». El **F1-score** resume precisión y recall mediante su media armónica.

Estas métricas solo miden la coincidencia con las **etiquetas sintéticas**; no permiten validar el supuesto comportamiento de una persona.

## Paso 10. Interpretación del índice Gini

El **índice de impureza Gini** indica qué tan mezcladas están las clases dentro de un nodo del árbol. No es una métrica de exactitud del modelo sobre los datos de prueba.

Para dos clases se calcula así:

\[
Gini = 1 - p_0^2 - p_1^2
\]

Donde \(p_0\) es la proporción de la clase 0 y \(p_1\) es la proporción de la clase 1 dentro de un nodo.

- **Gini = 0:** todos los registros del nodo pertenecen a una sola clase; el nodo es puro.
- **Gini = 0,5:** hay un reparto 50 % / 50 %; es la mayor impureza posible con dos clases.
- **Gini = 0,18:** por ejemplo, una mezcla 90 % / 10 %; es un nodo relativamente puro.

**Ejemplo numérico:** si un nodo tiene 8 registros fieles y 2 infieles, entonces:

\[
Gini = 1-(8/10)^2-(2/10)^2 = 0,32
\]

Durante el entrenamiento, el árbol busca divisiones que reduzcan la **impureza Gini ponderada de los nodos hijos**. Un Gini bajo no garantiza que el modelo generalice bien a casos nuevos; por ello también se evalúa con datos de prueba.

In [ ]:
p_fiel = 8 / 10
p_infiel = 2 / 10
gini_ejemplo = 1 - p_fiel**2 - p_infiel**2

print(f"Gini del ejemplo (8 fieles, 2 infieles): {gini_ejemplo:.2f}")
print(f"Gini del nodo raíz del árbol entrenado: {modelo.tree_.impurity[0]:.3f}")

## Paso 11. ¿Qué variables utiliza más el árbol?

Se calcula la **importancia de las características**. En un árbol de decisión, este valor resume cuánto contribuyó cada variable a reducir la impureza durante el entrenamiento.

Una importancia alta **no prueba causalidad**, y una importancia baja no significa que la variable sea irrelevante fuera de este ejercicio. Los resultados dependen de las etiquetas y de las decisiones tomadas al construir el árbol.

In [ ]:
importancias = pd.Series(modelo.feature_importances_, index=variables)
importancias = importancias.sort_values(ascending=False)

display(importancias.to_frame("importancia"))
importancias.sort_values().plot(kind="barh", figsize=(9, 5))
plt.xlabel("Importancia en el árbol")
plt.title("Importancia de las variables")
plt.tight_layout()
plt.show()

## Paso 12. Uso del Modelo

Las variables binarias se ingresan como `1` para **sí** y `0` para **no**. Las demás reciben cantidades numéricas. Después, se construye una fila con exactamente las mismas columnas utilizadas al entrenar..

In [ ]:
# Los alumnos modifican estos valores para probar distintos escenarios.
alumno = {
    "edad": 38,
    "numero_relaciones_anteriores": 7,
    "horas_sin_responder": 12,
    "oculta_celular": 1,
    "tiene_excusas": 1,
    "le_gusta_regueton": 1,
    "le_gustan_videojuegos": 1,
    "horas_videojuegos_semana": 10,
    "ve_anime": 1,
    "programa_fines_de_semana": 1,
    "salidas_fiesta_mensual": 0
}

nuevo_alumno = pd.DataFrame([alumno], columns=variables)
resultado = int(modelo.predict(nuevo_alumno)[0])
proporciones = modelo.predict_proba(nuevo_alumno)[0]

print("Clasificación ficticia:", "Infiel (1)" if resultado == 1 else "Fiel (0)")
print(f"Proporción asignada a Fiel (0): {proporciones[0]:.1%}")
print(f"Proporción asignada a Infiel (1): {proporciones[1]:.1%}")